In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

orders = pd.read_excel("List_of_Orders.xlsx")
details = pd.read_excel("Order_Details.xlsx")
target = pd.read_excel("Sales_target.xlsx")

orders.head()

In [ ]:
orders["Order Date"] = pd.to_datetime(orders["Order Date"], dayfirst=True, errors="coerce")
orders = orders.dropna(subset=["Order ID"])
details = details.dropna(subset=["Order ID"])
orders["Order Date"].head()

In [ ]:
df = details.merge(orders, on="Order ID", how="left")
print(df.shape)
df.head()

In [ ]:
cat = (
    df.groupby("Category")
    .agg(
        Total_Sales=("Amount", "sum"),
        Total_Profit=("Profit", "sum"),
        Orders=("Order ID", "nunique"),
    )
    .reset_index()
)
cat["Avg_Profit_per_Order"] = cat["Total_Profit"] / cat["Orders"]
cat["Profit_Margin_%"] = (cat["Total_Profit"] / cat["Total_Sales"] * 100).round(2)
cat.sort_values("Total_Sales", ascending=False)

In [ ]:
fig, ax1 = plt.subplots(figsize=(7, 4.5))
ax1.bar(cat["Category"], cat["Total_Sales"], color="#A9CCE3")
ax1.set_ylabel("Total Sales (₹)")

ax2 = ax1.twinx()
ax2.plot(cat["Category"], cat["Profit_Margin_%"], color="red", marker="o")
ax2.set_ylabel("Profit Margin (%)")

plt.title("Sales vs Profit Margin by Category")
plt.show()

In [ ]:
target["Month of Order Date"] = pd.to_datetime(target["Month of Order Date"])

def fix_month(d):
    year = 2000 + d.day
    return pd.Timestamp(year=year, month=d.month, day=1)

target["Period"] = target["Month of Order Date"].apply(fix_month)
target.head()

In [ ]:
furniture_target = target[target["Category"] == "Furniture"].sort_values("Period").reset_index(drop=True)
furniture_target["MoM_Change_%"] = furniture_target["Target"].pct_change().round(4) * 100
furniture_target["Month_Label"] = furniture_target["Period"].dt.strftime("%b-%Y")
furniture_target[["Month_Label", "Target", "MoM_Change_%"]]

In [ ]:
furn_actual = df[df["Category"] == "Furniture"].copy()
furn_actual["Period"] = furn_actual["Order Date"].dt.to_period("M").dt.to_timestamp()
furn_actual_m = furn_actual.groupby("Period")["Amount"].sum().reset_index(name="Actual")

furn_compare = furniture_target.merge(furn_actual_m, on="Period", how="left")
furn_compare["Achievement_%"] = (furn_compare["Actual"] / furn_compare["Target"] * 100).round(1)
furn_compare[["Month_Label", "Target", "Actual", "Achievement_%"]]

In [ ]:
state_orders = orders.groupby("State")["Order ID"].nunique().reset_index(name="Order_Count")
top5_states = state_orders.sort_values("Order_Count", ascending=False).head(5)["State"].tolist()
print(top5_states)

state_perf = (
    df[df["State"].isin(top5_states)]
    .groupby("State")
    .agg(Order_Count=("Order ID", "nunique"), Total_Sales=("Amount", "sum"), Total_Profit=("Profit", "sum"))
    .reset_index()
)
state_perf["Avg_Profit_per_Order"] = (state_perf["Total_Profit"] / state_perf["Order_Count"]).round(2)
state_perf.sort_values("Order_Count", ascending=False)

In [ ]:
city_perf = (
    df[df["State"].isin(top5_states)]
    .groupby(["State", "City"])
    .agg(Order_Count=("Order ID", "nunique"), Total_Sales=("Amount", "sum"), Total_Profit=("Profit", "sum"))
    .reset_index()
)
city_perf["Profit_Margin_%"] = (city_perf["Total_Profit"] / city_perf["Total_Sales"] * 100).round(2)
city_perf.sort_values(["State", "Total_Sales"], ascending=[True, False])